<a href="https://colab.research.google.com/github/AzadMahmud/dl-lab/blob/main/assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

ROOT = Path(os.environ.get("ASSIGN3_ROOT", SCRIPT_DIR.parent)).resolve()
IMAGE_PATH = ROOT / "self_image.jpg"
if not IMAGE_PATH.exists() and Path("/content/self_image.jpg").exists():
    IMAGE_PATH = Path("/content/self_image.jpg")

OUT = ROOT / "outputs"
FMAP_DIR = OUT / "feature_maps"
CURVE_DIR = OUT / "training_curves"
METRIC_DIR = OUT / "metrics"
MODEL_DIR = OUT / "models"

for d in [OUT, FMAP_DIR, CURVE_DIR, METRIC_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:

def save_feature_maps(image_path: Path) -> None:
    base = tf.keras.applications.VGG16(weights="imagenet", include_top=False)
    chosen_layers = [
        "block1_conv1",
        "block1_conv2",
        "block2_conv1",
        "block3_conv1",
        "block4_conv1",
        "block5_conv1",
    ]
    feat_model = tf.keras.Model(
        inputs=base.input,
        outputs=[base.get_layer(name).output for name in chosen_layers],
    )

    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    x = tf.keras.utils.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = tf.keras.applications.vgg16.preprocess_input(x)

    feature_maps = feat_model.predict(x, verbose=0)

    summary_rows = []
    for layer_name, fmap in zip(chosen_layers, feature_maps):
        fmap = fmap[0]
        h, w, c = fmap.shape
        n_show = min(16, c)

        fig, axes = plt.subplots(4, 4, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            if i < n_show:
                ch = fmap[:, :, i]
                ch = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
                ax.imshow(ch, cmap="viridis")
                ax.set_title(f"ch {i}", fontsize=8)
            ax.axis("off")
        fig.suptitle(f"{layer_name} ({h}x{w}x{c})", fontsize=11)
        fig.tight_layout()
        fig.savefig(FMAP_DIR / f"{layer_name}.png", dpi=220)
        plt.close(fig)

        summary_rows.append(
            {
                "layer": layer_name,
                "height": int(h),
                "width": int(w),
                "channels": int(c),
            }
        )

    pd.DataFrame(summary_rows).to_csv(METRIC_DIR / "feature_map_shapes.csv", index=False)


In [ ]:
def activation_block(x, name: str):
    if name == "relu":
        return layers.Activation("relu")(x)
    if name == "elu":
        return layers.Activation("elu")(x)
    if name == "leaky_relu":
        return layers.LeakyReLU(negative_slope=0.1)(x)
    raise ValueError(f"Unsupported activation: {name}")


In [ ]:
def make_custom_cnn(activation: str) -> tf.keras.Model:
    inp = layers.Input(shape=(32, 32, 3))

    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal")(inp)
    x = layers.BatchNormalization()(x)
    x = activation_block(x, activation)
    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal")(x)
    x = activation_block(x, activation)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = activation_block(x, activation)
    x = layers.SeparableConv2D(64, 3, padding="same", depthwise_initializer="he_normal", pointwise_initializer="he_normal")(x)
    x = activation_block(x, activation)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal", dilation_rate=1)(x)
    x = layers.BatchNormalization()(x)
    x = activation_block(x, activation)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, kernel_initializer="he_normal")(x)
    x = activation_block(x, activation)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(10, activation="softmax")(x)

    model = models.Model(inp, out, name=f"custom_cnn_{activation}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
def load_cifar10():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0
    y_train = tf.keras.utils.to_categorical(y_train, 10)
    y_test = tf.keras.utils.to_categorical(y_test, 10)

    x_val, y_val = x_train[-5000:], y_train[-5000:]
    x_train, y_train = x_train[:-5000], y_train[:-5000]
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)


In [ ]:
def make_dataset(x, y, training: bool, batch_size: int = 128):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        aug = tf.keras.Sequential(
            [
                layers.RandomFlip("horizontal"),
                layers.RandomTranslation(0.1, 0.1),
            ]
        )
        ds = ds.shuffle(10000, seed=SEED)
        ds = ds.map(lambda a, b: (aug(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
def plot_histories(histories: dict[str, dict[str, list[float]]]):
    plt.figure(figsize=(10, 4))
    for name, h in histories.items():
        plt.plot(h["val_accuracy"], label=f"{name} val_acc")
    plt.title("Validation Accuracy Across Activations")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(CURVE_DIR / "val_accuracy_compare.png", dpi=220)
    plt.close()

    plt.figure(figsize=(10, 4))
    for name, h in histories.items():
        plt.plot(h["val_loss"], label=f"{name} val_loss")
    plt.title("Validation Loss Across Activations")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(CURVE_DIR / "val_loss_compare.png", dpi=220)
    plt.close()


In [ ]:
def train_activation_comparison(epochs: int = 12):
    (x_train, y_train), (x_val, y_val), (x_test, y_test) = load_cifar10()
    ds_train = make_dataset(x_train, y_train, training=True)
    ds_val = make_dataset(x_val, y_val, training=False)
    ds_test = make_dataset(x_test, y_test, training=False)

    activations = ["relu", "elu", "leaky_relu"]
    rows = []
    histories = {}

    for act in activations:
        print(f"Training activation: {act}")
        model = make_custom_cnn(act)
        cbs = [
            tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=3, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
        ]

        t0 = time.time()
        hist = model.fit(ds_train, validation_data=ds_val, epochs=epochs, verbose=1, callbacks=cbs)
        t1 = time.time()
        test_loss, test_acc = model.evaluate(ds_test, verbose=0)

        h = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
        histories[act] = h
        pd.DataFrame(h).to_csv(METRIC_DIR / f"history_{act}.csv", index=False)
        model.save(MODEL_DIR / f"custom_cnn_{act}.keras")

        rows.append(
            {
                "activation": act,
                "epochs_ran": len(h["loss"]),
                "best_val_accuracy": max(h["val_accuracy"]),
                "final_val_accuracy": h["val_accuracy"][-1],
                "final_train_accuracy": h["accuracy"][-1],
                "test_accuracy": float(test_acc),
                "test_loss": float(test_loss),
                "train_time_sec": float(t1 - t0),
            }
        )

        tf.keras.backend.clear_session()

    metrics_df = pd.DataFrame(rows).sort_values("test_accuracy", ascending=False)
    metrics_df.to_csv(METRIC_DIR / "activation_comparison.csv", index=False)
    plot_histories(histories)

    best = metrics_df.iloc[0].to_dict()
    with open(METRIC_DIR / "summary.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "best_activation": best["activation"],
                "best_test_accuracy": best["test_accuracy"],
                "note": "Compare convergence from val_accuracy_compare.png and val_loss_compare.png",
            },
            f,
            indent=2,
        )

In [ ]:
def main():
    if not IMAGE_PATH.exists():
        raise FileNotFoundError(f"self_image.jpg not found at {IMAGE_PATH}")

    save_feature_maps(IMAGE_PATH)
    train_activation_comparison(epochs=12)
    print(f"Done. Outputs saved under: {OUT}")


if __name__ == "__main__":
    main()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step
Training activation: relu
Epoch 1/12
352/352 ━━━━━━━━━━━━━━━━━━━━ 57s 126ms/step - accuracy: 0.3811 - loss: 1.6592 - val_accuracy: 0.4326 - val_loss: 1.5548 - learning_rate: 0.0010
Epoch 2/12
352/352 ━━━━━━━━━━━━━━━━━━━━ 65s 100ms/step - accuracy: 0.5307 - loss: 1.2965 - val_accuracy: 0.5736 - val_loss: 1.1928 - learning_rate: 0.0010
Epoch 3/12
352/352 ━━━━━━━━━━━━━━━━━━━━ 35s 100ms/step - accuracy: 0.5796 - loss: 1.1728 - val_accuracy: 0.5806 - val_loss: 1.1586 - learning_rate: 0.0010
Epoch 4/12
352/352 ━━━━━━━━━━━━━━━━━━━━ 35s 99ms/step - accuracy: 0.6144 - loss: 1.0857 - val_accuracy: 0.5886 - val_loss: 1.2126 - learning_rate: 0.0010
Epoch 5/12
352/352 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.6359 - loss: 1.0319 - val_accuracy: 0.6174 - val_loss: 1.0726 - learning_rate: 0.0010
Epoch 6/12
352/352 ━━━━━━━━━━━━━━━━━━━━ 42s 99ms/step - accuracy: 0.6497 - loss: 0.9913 - val_acc